# Experiment

**Hypothesis:** ...

**Change from baseline:** ...

**Starting checkpoint (if transferring):** ...

**Comparison and success metric:** ...

This is a clean experiment starter with a working SchNet configuration. Select this folder's `.venv/bin/python` as the local kernel. After duplicating or moving the local template, run `bash setup-local.sh` in the new folder first.

The fixed data split is 800 training / 50 validation / 150 test structures. `data/raw.xyz` is the original 850-structure source, not an additional training input. The split is random by structure, not verified by parent simulation. Keep the test data for final evaluation.


In [ ]:
from pathlib import Path
from importlib.metadata import version
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ase.io
import torch

project = Path.cwd()
assert (project / "train.yaml").is_file(), "Open the notebook from the experiment folder."
print("Python:", sys.executable)
print("graph-pes:", version("graph-pes"), "PyTorch:", torch.__version__)


In [ ]:
training = ase.io.read(project / "data/train.xyz", index=":")
validation = ase.io.read(project / "data/valid.xyz", index=":")
print(f"Training: {len(training)}; validation: {len(validation)}")
assert len(training) == 800 and len(validation) == 50
print("Atoms:", len(training[0]), "Periodic:", training[0].pbc)
print("Energy:", training[0].info["QM_energy"])
print("Forces:", training[0].arrays["QM_forces"].shape)


## Small local check

Edit `train.yaml` to configure the experiment, then run this cell. It checks eight training and eight validation structures with two batches for one epoch on CPU. This is a setup check, not a useful accuracy benchmark.

The command starts a separate process that reads the YAML. Python variables or model changes made elsewhere in this notebook are not automatically passed into that process.


In [ ]:
import subprocess

train_command = Path(sys.executable).parent / "graph-pes-train"
subprocess.run([
    str(train_command), "train.yaml",
    "general/run_id=local-check",
    "data/train/n=8", "data/valid/n=8",
    "fitting/trainer_kwargs/accelerator=cpu",
    "fitting/trainer_kwargs/max_epochs=1",
    "fitting/trainer_kwargs/limit_train_batches=2",
    "fitting/trainer_kwargs/limit_val_batches=2",
], cwd=project, check=True)


## Results and next step

Record observations and plots below.

For a full GPU run, copy the updated `train.yaml` to the matching experiment folder on ARC and submit `sbatch train.sbatch` there. You can copy this notebook too, but the default job executes `graph-pes-train train.yaml`, not the notebook. For custom Python training logic (such as freezing layers), that logic must be in code executed by the training entry point; editing an unrelated notebook cell alone will not change the ARC run.

The ARC template already contains the same data and reuses the existing ARC environment. Download results into this local experiment for analysis. Local CPU and ARC GPU runs need not be numerically identical.
